# <center>Semantic Distance in thesaurus data</center>

```AQL
FOR node
IN ANY ALL_SHORTEST_PATHS
@ID_1 TO @ID_2
GRAPH @Graph_name
OPTIONS {
  weightAttribute : "weight"
}
RETURN node
```

In [1]:
from multiprocessing.forkserver import connect_to_new_process

from arango import ArangoClient, database
from random import randint

client = ArangoClient(hosts="http://localhost:8529")
db : database.StandardDatabase = client.db("TEATIME", username="root", password="test")
thesaurus_documents : database.StandardCollection = db.collection("th15")
graph_name = "th15_graph"

In [2]:
documents_cursor = thesaurus_documents.all()
thesaurus_doc_id_as_list : list = [doc['_id'] for doc in documents_cursor] # Pour choisir deux concepts au hasard dans notre query
print(thesaurus_doc_id_as_list[:10])

['th15/12f57232-ccb0-4cd8-8962-8133586b80aa', 'th15/18748ed6-fdb0-4f3a-a4ab-991ba5accf44', 'th15/4a47d11d-68a9-436f-a9da-20cd993439b0', 'th15/53d7c9ce-6703-42ea-a00d-9430d52a109d', 'th15/76306469-4255-4228-9484-6aeef4af1807', 'th15/868bac34-b90d-4189-a8f8-7148e7c9fd70', 'th15/964898d2-66e1-43c9-801b-e43e9eaf2f93', 'th15/T69-1001', 'th15/T69-1007', 'th15/T69-1009']


In [4]:
id1 = thesaurus_doc_id_as_list[randint(0, len(thesaurus_doc_id_as_list))]
id2 = thesaurus_doc_id_as_list[randint(0, len(thesaurus_doc_id_as_list))]

while id1 == id2:
    id2 = thesaurus_doc_id_as_list[randint(0, len(thesaurus_doc_id_as_list))]

result = db.aql.execute("FOR node \
                IN ANY SHORTEST_PATH \
                @ID_1 TO @ID_2 \
                GRAPH @Graph_name \
                OPTIONS { \
                  weightAttribute : 'weight' \
                }\
                RETURN node.name",
               bind_vars={"ID_1": id1, "ID_2" : id2, "Graph_name" : graph_name})

print(result.next(), end='')

for thing in result:
    print(f" --> {thing}", end='')



mesure de référence --> outil servant à mesurer ou dessiner --> outil --> matériel artisanal --> matériel de manutention --> charrette à bras --> véhicule terrestre à énergie naturelle ou humaine --> véhicule terrestre non motorisé --> matériel de transport terrestre --> moyen de transport --> matériel de transport ferroviaire --> matériel ferroviaire roulant

C'est cool mais est-ce qu'on pourrais pas faire mieux ? Par exemple avoir aussi chaque edge label


In [113]:
id1 = thesaurus_doc_id_as_list[randint(0, len(thesaurus_doc_id_as_list))]
id2 = thesaurus_doc_id_as_list[randint(0, len(thesaurus_doc_id_as_list))]

while id1 == id2:
    id2 = thesaurus_doc_id_as_list[randint(0, len(thesaurus_doc_id_as_list))]

result = db.aql.execute("FOR node, edge \
                IN ANY SHORTEST_PATH \
                @ID_1 TO @ID_2 \
                GRAPH @Graph_name \
                RETURN [node.name, edge.type, edge.weight]",
               bind_vars={"ID_1": id1, "ID_2" : id2, "Graph_name" : graph_name})

print(result.next()[0], end='')

nbEdge = 0
distance = 0

for thing in result:
    distance += thing[2]
    nbEdge += 1
    print(f" <--{thing[1]}--", end='')
    if thing[1] != "narrower" and thing[1] != "broader":
        print(f">", end='')

    print(f" {thing[0]}", end='')

print("")
print(f"Distance = {distance}, nombre de relations = {nbEdge}")

print("Et avec des poids :")

result = db.aql.execute("FOR node, edge \
                IN ANY SHORTEST_PATH \
                @ID_1 TO @ID_2 \
                GRAPH @Graph_name \
                OPTIONS { \
                  weightAttribute : 'weight' \
                }\
                RETURN [node.name, edge.type, edge.weight]",
               bind_vars={"ID_1": id1, "ID_2" : id2, "Graph_name" : graph_name})

print(result.next()[0], end='')

distance = 0
nbEdge = 0

for thing in result:
    distance += thing[2]
    nbEdge += 1
    print(f" <--{thing[1]}--", end='')
    if thing[1] != "narrower" and thing[1] != "broader":
        print(f">", end='')

    print(f" {thing[0]}", end='')

print("")
print(f"Distance = {distance}, nombre de relations = {nbEdge}")


plaque décorative <--narrower-- arts graphiques <--broader-- carte <--related--> objet de représentation de corps célestes <--narrower-- objet de représentation scientifique et pédagogique <--narrower-- objet pédagogique ou scientifique <--narrower-- instrument ou objet pédagogique ou scientifique <--broader-- instrument pédagogique ou scientifique <--broader-- instrument de calcul <--broader-- instrument de calcul mécanique
Distance = 11, nombre de relations = 9
Et avec des poids :
plaque décorative <--narrower-- arts graphiques <--broader-- carte <--related--> objet de représentation de corps célestes <--narrower-- objet de représentation scientifique et pédagogique <--narrower-- objet pédagogique ou scientifique <--narrower-- instrument ou objet pédagogique ou scientifique <--broader-- instrument pédagogique ou scientifique <--broader-- instrument de calcul <--broader-- instrument de calcul mécanique
Distance = 11, nombre de relations = 9


same with weights

In [122]:
result = db.aql.execute("FOR node, edge \
                IN ANY SHORTEST_PATH \
                @ID_1 TO @ID_2 \
                GRAPH @Graph_name \
                OPTIONS { \
                  weightAttribute : 'weight' \
                }\
                RETURN [node.name, edge.type, edge.weight]",
               bind_vars={"ID_1": id1, "ID_2" : id2, "Graph_name" : graph_name})

print(result.next()[0], end='')

distance = 0
nbEdge = 0

for thing in result:
    distance += thing[2]
    nbEdge += 1
    print(f" <--{thing[1]}--", end='')
    if thing[1] != "narrower" and thing[1] != "broader":
        print(f">", end='')

    print(f" {thing[0]}", end='')

print("")
print(f"Distance = {distance}, nombre de relations = {nbEdge}")

plaque décorative <--narrower-- arts graphiques <--broader-- carte <--related--> objet de représentation de corps célestes <--narrower-- objet de représentation scientifique et pédagogique <--narrower-- objet pédagogique ou scientifique <--narrower-- instrument ou objet pédagogique ou scientifique <--broader-- instrument pédagogique ou scientifique <--broader-- instrument de calcul <--broader-- instrument de calcul mécanique
Distance = 11, nombre de relations = 9


On va essayer de faire une matrice de distance sur un set restreint de 10 concepts randomisés

In [138]:
concepts : list[str] = []
nb_concepts = 10

if nb_concepts >= len(concepts):
    concepts = thesaurus_doc_id_as_list
else:
    for i in range(nb_concepts):
        concepts.append(thesaurus_doc_id_as_list[randint(0, len(thesaurus_doc_id_as_list))])
        while concepts[:i-1].__contains__(concepts[i]):
            concepts[i] = thesaurus_doc_id_as_list[randint(0, len(thesaurus_doc_id_as_list))]

matrix = [[0 for i in range(nb_concepts)] for i in range(nb_concepts)]

for i in range(nb_concepts):
    for j in range(i + 1, nb_concepts):
        if i != j :
            result = db.aql.execute("FOR node, edge \
                    IN ANY SHORTEST_PATH \
                    @ID_1 TO @ID_2 \
                    GRAPH @Graph_name \
                    OPTIONS { \
                      weightAttribute : 'weight' \
                    }\
                    RETURN [node.name, edge.weight]",
                   bind_vars={"ID_1": concepts[i], "ID_2" : concepts[j], "Graph_name" : graph_name})

            result.next() # We skip the first element, which is the current node and a None type element

            distance = 0

            for weight in result:
                distance += weight[1]

            matrix[i][j] = matrix[j][i] = distance

for line in matrix :
    print(line)


[0, 14, 14, 12, 13, 5, 7, 16, 16, 16]
[14, 0, 2, 17, 5, 15, 17, 13, 13, 13]
[14, 2, 0, 17, 5, 15, 17, 13, 13, 13]
[12, 17, 17, 0, 16, 11, 11, 13, 13, 13]
[13, 5, 5, 16, 0, 14, 16, 12, 12, 12]
[5, 15, 15, 11, 14, 0, 4, 14, 14, 14]
[7, 17, 17, 11, 16, 4, 0, 15, 15, 15]
[16, 13, 13, 13, 12, 14, 15, 0, 2, 2]
[16, 13, 13, 13, 12, 14, 15, 2, 0, 2]
[16, 13, 13, 13, 12, 14, 15, 2, 2, 0]


On remarquera facilement que l'on a énormément d'égalité entre différentes distances entre concepts, ce qui est attendu. Des tests sur les distances entre sets de termes pourront déterminer si il est utile de désambiguer. On va aussi avoir besoin de normaliser tout ça pour essayer de l'utiliser dans du clustering hierarchique.